<a href="https://colab.research.google.com/github/jsansao/teic-20231/blob/main/Licao37_llms_api_engenharia_de_prompt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Guia Prático de LLMs via API e Engenharia de Prompt (NVIDIA NIM)

Este notebook apresenta, de forma prática e didática, como interagir com **Grandes Modelos de Linguagem (LLMs)** por meio de APIs utilizando os endpoints gratuitos da **NVIDIA NIM** (compatíveis com o SDK oficial da OpenAI).

---

### Roteiro do Notebook

1. **Configuração e Autenticação na API da NVIDIA NIM**
   - Leitura segura da chave em `chave.txt` e inicialização do cliente `OpenAI`.
2. **Parte 1 — Anatomia de um Prompt, Parâmetros e Janela de Contexto**
   - **1.1 Anatomia de um Prompt**: Os 4 blocos construtivos (Instrução, Contexto, Dados de Entrada e Indicador de Saída) e inspeção dos metadados da resposta (`usage`, `finish_reason`).
   - **1.2 Papéis das Mensagens (`system`, `user`, `assistant`)**: Como estruturar conversas na API.
   - **1.3 Parâmetros Configuráveis**: `temperature`, `top_p`, `max_tokens`, `stop`, penalidades e `stream`.
   - **1.4 Janela de Contexto e Múltiplas Trocas de Mensagem**: A natureza *stateless* das APIs, acúmulo do histórico, crescimento do consumo de tokens e estratégias de gerenciamento de memória (janela deslizante).
3. **Parte 2 — Estratégias de Engenharia de Prompt**
   - **2.1 Zero-Shot Prompting**
   - **2.2 One-Shot Prompting**
   - **2.3 Few-Shot Prompting**
   - **2.4 Role Prompting / System Persona**
   - **2.5 Saída Estruturada / JSON Mode**
   - **2.6 Chain-of-Thought (CoT — Cadeia de Pensamento)**
   - **2.7 Modelos de Reasoning (Raciocínio Nativo / `reasoning_content`)**
   - **2.8 Self-Consistency e Reflexão (Autoavaliação e Refinamento)**
   - **2.9 Prompt Chaining e Padrão ReAct (Reason + Act)**

## 0. Configuração do Ambiente e Autenticação na NVIDIA NIM

A **NVIDIA NIM (NVIDIA Inference Microservices)** disponibiliza uma API compatível com o padrão **OpenAI Chat Completions**. Isso significa que podemos utilizar a biblioteca oficial `openai` apenas alterando o parâmetro `base_url` para:

```
https://integrate.api.nvidia.com/v1
```

Abaixo, carregamos a chave de API armazenada no arquivo `chave.txt` do diretório atual e configuramos os modelos que usaremos ao longo do notebook:
- **Modelo Padrão (Instruct rápido):** `nvidia/nemotron-3.5-bolt-e4b` — ideal para tarefas gerais de instrução, classificação, extração e conversação com baixíssima latência.
- **Modelo de Reasoning (Raciocínio explícito):** `nvidia/nemotron-3-ultra-550b-a55b` — modelo que gera tokens dedicados de pensamento interno (`reasoning_content`) antes de produzir a resposta final.

In [1]:
from pathlib import Path
import json
import time
from openai import OpenAI


def carregar_chave_nim(caminho_arquivo: str = "chave.txt") -> str:
    """Lê a chave de API do arquivo chave.txt (suporta 'CHAVE = valor' ou apenas o valor)."""
    conteudo = Path(caminho_arquivo).read_text(encoding="utf-8").strip()
    if "=" in conteudo:
        return conteudo.split("=", 1)[1].strip()
    return conteudo


API_KEY = carregar_chave_nim("chave.txt")

# Inicializa o cliente apontando para o endpoint da NVIDIA NIM
client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=API_KEY,
)

# Modelos selecionados na NVIDIA NIM
MODELO_PADRAO = "nvidia/nemotron-3.5-bolt-e4b"
MODELO_REASONING = "nvidia/nemotron-3-ultra-550b-a55b"

print(f"Chave carregada com sucesso! Prefixo: {API_KEY[:10]}... (tamanho: {len(API_KEY)} caracteres)")
print(f"Modelo padrão configurado   : {MODELO_PADRAO}")
print(f"Modelo reasoning configurado: {MODELO_REASONING}")

Chave carregada com sucesso! Prefixo: nvapi-2gUS... (tamanho: 70 caracteres)
Modelo padrão configurado   : nvidia/nemotron-3.5-bolt-e4b
Modelo reasoning configurado: nvidia/nemotron-3-ultra-550b-a55b


---
# Parte 1: Anatomia de um Prompt, Parâmetros e Janela de Contexto

## 1.1 Anatomia de um Prompt

Um prompt bem estruturado geralmente é composto por até **quatro elementos fundamentais**:

| Componente | Função | Exemplo |
| :--- | :--- | :--- |
| **1. Instrução (Tarefa)** | Define claramente o que o modelo deve fazer (verbo de ação). | *"Classifique a urgência do chamado técnico abaixo e sugira uma ação inicial."* |
| **2. Contexto** | Informações externas, regras de negócio ou restrições que guiam a resposta. | *"Considere que indisponibilidade em produção é Crítica, lentidão é Média e dúvidas visuais são Baixas."* |
| **3. Dados de Entrada (Input)** | O conteúdo concreto a ser processado, idealmente delimitado por tags ou marcadores (`"""`, `<texto>`, `---`). | *"<chamado>O sistema de pagamentos retorna erro 500 para todos os clientes desde as 14h.</chamado>"* |
| **4. Indicador / Formato de Saída** | Especifica o formato, tamanho, idioma ou estrutura esperada na resposta. | *"Responda apenas com os campos: Nível de Urgência, Justificativa (1 frase) e Próximo Passo."* |

Além do texto gerado, toda chamada à API retorna um objeto estruturado (`ChatCompletion`) contendo metadados essenciais sobre o consumo de tokens (`usage`) e o motivo de encerramento da geração (`finish_reason`).

In [2]:
# Construindo um prompt com os 4 componentes da anatomia claramente delimitados
prompt_anatomia = """
[INSTRUÇÃO]
Analise o relato de suporte técnico abaixo, classifique seu nível de prioridade e indique a equipe responsável.

[CONTEXTO / REGRAS DE NEGÓCIO]
- Prioridade CRÍTICA: sistema fora do ar, falha em pagamentos ou perda de dados (Equipe: SRE / Infraestrutura).
- Prioridade MÉDIA: erro parcial com alternativa disponível ou lentidão (Equipe: Sustentação Backend).
- Prioridade BAIXA: dúvidas de uso ou ajustes visuais (Equipe: Atendimento N1 / Frontend).

[DADOS DE ENTRADA]
<relato>
Estamos tentando finalizar compras com cartão de crédito no checkout, mas a API retorna Timeout após 30 segundos para 100% dos usuários.
</relato>

[FORMATO DE SAÍDA]
Responda exatamente neste formato:
- Prioridade: <nível>
- Equipe Responsável: <equipe>
- Resumo Técnico (máx. 15 palavras): <resumo>
""".strip()

resposta = client.chat.completions.create(
    model=MODELO_PADRAO,
    messages=[{"role": "user", "content": prompt_anatomia}],
    temperature=0.0,
)

# Exibindo o texto da resposta e inspecionando a anatomia do objeto retornado pela API
escolha = resposta.choices[0]
print("=== TEXTO GERADO PELO MODELO ===")
print(escolha.message.content)

print("\n=== METADADOS DA RESPOSTA (ANATOMIA DO RETORNO DA API) ===")
print(f"ID da requisição     : {resposta.id}")
print(f"Modelo utilizado     : {resposta.model}")
print(f"Motivo de parada     : {escolha.finish_reason}  ('stop' = concluiu naturalmente, 'length' = atingiu max_tokens)")
print(f"Tokens de entrada    : {resposta.usage.prompt_tokens}")
print(f"Tokens gerados       : {resposta.usage.completion_tokens}")
print(f"Total de tokens      : {resposta.usage.total_tokens}")

=== TEXTO GERADO PELO MODELO ===
- Prioridade: CRÍTICA
- Equipe Responsável: SRE / Infraestrutura
- Resumo Técnico: Falha de timeout na API de checkout com cartão de crédito afetando todos os usuários.

=== METADADOS DA RESPOSTA (ANATOMIA DO RETORNO DA API) ===
ID da requisição     : chatcmpl-d1cfa612-351f-428a-8bbc-7000fca54a70
Modelo utilizado     : nvidia/nemotron-3.5-bolt-e4b
Motivo de parada     : stop  ('stop' = concluiu naturalmente, 'length' = atingiu max_tokens)
Tokens de entrada    : 236
Tokens gerados       : 44
Total de tokens      : 280


## 1.2 As Mensagens Passadas à API e seus Papéis (`roles`)

Na API de *Chat Completions*, não enviamos apenas uma string isolada, mas sim uma **lista ordenada de mensagens** (`messages`), onde cada item possui um `role` (papel) e um `content` (conteúdo):

1. **`system`**: Define as instruções globais, persona, tom de voz, regras de segurança e restrições de formato. Funciona como o "diretor de cena" do modelo.
2. **`user`**: Representa as perguntas, comandos ou dados enviados pelo usuário final (ou pela aplicação).
3. **`assistant`**: Representa as respostas anteriores geradas pelo próprio LLM (usado para manter o histórico da conversa ou fornecer exemplos *few-shot*).

Veja abaixo como **a mesma mensagem de `user`** produz respostas completamente diferentes apenas alterando a mensagem de `system`:

In [3]:
pergunta_usuario = "O que é recursão em programação?"

sistemas = [
    ("Professor para Crianças", "Você explica conceitos de computação para crianças de 8 anos usando analogias lúdicas em até 3 frases."),
    ("Engenheiro de Software Sênior", "Você é um arquiteto de sistemas especialista em baixo nível. Explique focando em pilha de chamadas (call stack), stack overflow e complexidade espacial em até 3 frases."),
]

for nome_perfil, instrucao_sistema in sistemas:
    resp = client.chat.completions.create(
        model=MODELO_PADRAO,
        messages=[
            {"role": "system", "content": instrucao_sistema},
            {"role": "user", "content": pergunta_usuario},
        ],
        temperature=0.2,
    )
    print(f"=== PERFIL DO SYSTEM: {nome_perfil} ===")
    print(resp.choices[0].message.content.strip())
    print()

=== PERFIL DO SYSTEM: Professor para Crianças ===
Imagina um espelho dentro de outro espelho: cada um reflete o outro! A recursão é quando uma função chama a si mesma, como um boneco que faz um boneco menor que faz outro ainda menor, até chegar ao fim. Assim, resolvemos problemas grandes dividindo-os em pedacinhos iguais.

=== PERFIL DO SYSTEM: Engenheiro de Software Sênior ===
Recursão utiliza uma função que chama a si mesma, gerenciando o estado através da **pilha de chamadas** (call stack). O **stack overflow** ocorre quando a profundidade da recursão excede o limite de memória alocado para a stack. Isso implica uma **complexidade espacial** linear ($O(n)$) na profundidade da pilha, dependendo do número de chamadas.



## 1.3 Tipos de Parâmetros Configuráveis na Chamada da API

Ao invocar `client.chat.completions.create(...)`, podemos controlar a distribuição de probabilidade dos próximos tokens e os limites de geração:

| Parâmetro | Tipo / Faixa Comum | Descrição Prática | Quando Usar |
| :--- | :--- | :--- | :--- |
| **`temperature`** | `float` (`0.0` a `2.0`) | Controla a aleatoriedade na escolha dos tokens. Valores próximos de `0.0` tornam o modelo determinístico (escolhe sempre os tokens mais prováveis); valores altos (`0.8+`) achatam a distribuição e aumentam a criatividade/variedade. | `0.0` para extração de dados, código, classificação e JSON; `0.7–1.0` para brainstorming e escrita criativa. |
| **`top_p`** *(Nucleus Sampling)* | `float` (`0.0` a `1.0`) | Considera apenas o menor conjunto de tokens cuja soma de probabilidades atinge `top_p` (ex.: `0.9` = 90% da massa de probabilidade). | Alternativa à temperatura. Recomenda-se ajustar `temperature` **ou** `top_p`, não ambos simultaneamente. |
| **`max_tokens`** | `int` | Limite máximo de tokens que o modelo pode gerar na resposta (`completion_tokens`). | Controle de custos, latência e prevenção de respostas excessivamente longas. |
| **`stop`** | `str` ou `list[str]` | Sequência(s) de texto que, ao serem geradas, interrompem imediatamente a resposta. | Parar a geração ao fechar uma tag XML/JSON ou impedir que o modelo continue gerando diálogos extras. |
| **`presence_penalty`** | `float` (`-2.0` a `2.0`) | Penaliza tokens que já apareceram no texto gerado (independentemente da frequência), incentivando o modelo a introduzir novos tópicos. | Evitar que o modelo fique preso no mesmo assunto em textos longos. |
| **`frequency_penalty`** | `float` (`-2.0` a `2.0`) | Penaliza tokens proporcionalmente à quantidade de vezes que já foram repetidos. | Reduzir repetição literal de palavras ou frases. |
| **`stream`** | `bool` | Quando `True`, retorna os pedaços (*chunks*) da resposta via Server-Sent Events (SSE) conforme são gerados. | Interfaces de chat em tempo real para reduzir o tempo percebido até a primeira palavra (*Time to First Token*). |

### Experimento A: Efeito da `temperature` (`0.0` vs `1.0`)

In [4]:
prompt_criativo = "Crie um slogan curto (máximo 8 palavras) e inédito para uma cafeteria espacial em Marte."

print("--- Temperature = 0.0 (Determinístico / Conservador) ---")
for i in range(3):
    r = client.chat.completions.create(
        model=MODELO_PADRAO,
        messages=[{"role": "user", "content": prompt_criativo}],
        temperature=0.0,
        max_tokens=40,
    )
    print(f"Execução {i+1}: {r.choices[0].message.content.strip()}")

print("\n--- Temperature = 1.0 (Criativo / Diversificado) ---")
for i in range(3):
    r = client.chat.completions.create(
        model=MODELO_PADRAO,
        messages=[{"role": "user", "content": prompt_criativo}],
        temperature=1.0,
        top_p=0.95,
        max_tokens=40,
    )
    print(f"Execução {i+1}: {r.choices[0].message.content.strip()}")

--- Temperature = 0.0 (Determinístico / Conservador) ---
Execução 1: **Café marciano: Sabor cósmico, pausa em Marte.**
Execução 2: **Café marciano: Sabor cósmico, pausa em Marte.**
Execução 3: **Café marciano: Sabor cósmico, pausa em Marte.**

--- Temperature = 1.0 (Criativo / Diversificado) ---
Execução 1: Marte: Seu café cósmico, sabor em cada gole.
Execução 2: **Café marciano: Sabor interplanetário, pausa espacial.**
Execução 3: **Café marciano: Sabor interplanetário, pausa espacial.**


### Experimento B: Controle de Parada (`max_tokens` e `stop`) e Streaming (`stream=True`)

In [5]:
# 1. Demonstração do parâmetro 'stop': parar assim que chegar no item 4 da lista
r_stop = client.chat.completions.create(
    model=MODELO_PADRAO,
    messages=[{"role": "user", "content": "Liste 10 linguagens de programação numeradas de 1. a 10., uma por linha."}],
    temperature=0.0,
    stop=["4."],
)
print("=== USO DE stop=['4.'] (interrompe antes do 4º item) ===")
print(r_stop.choices[0].message.content.strip())
print(f"finish_reason: {r_stop.choices[0].finish_reason}")

# 2. Demonstração de 'max_tokens' curto (corte por limite de tokens -> finish_reason = 'length')
r_len = client.chat.completions.create(
    model=MODELO_PADRAO,
    messages=[{"role": "user", "content": "Explique detalhadamente a história da inteligência artificial desde 1950."}],
    temperature=0.0,
    max_tokens=20,
)
print("\n=== USO DE max_tokens=20 (truncamento por limite de tokens) ===")
print(r_len.choices[0].message.content.strip())
print(f"finish_reason: {r_len.choices[0].finish_reason} (tokens gerados: {r_len.usage.completion_tokens})")

# 3. Demonstração de Streaming (stream=True)
print("\n=== USO DE stream=True (recebendo chunks em tempo real) ===")
stream = client.chat.completions.create(
    model=MODELO_PADRAO,
    messages=[{"role": "user", "content": "Escreva um haicai sobre inteligência artificial."}],
    temperature=0.3,
    stream=True,
)
chunks_recebidos = 0
for chunk in stream:
    if chunk.choices and chunk.choices[0].delta.content:
        print(chunk.choices[0].delta.content, end="", flush=True)
        chunks_recebidos += 1
print(f"\n[Streaming concluído em {chunks_recebidos} chunks]")

=== USO DE stop=['4.'] (interrompe antes do 4º item) ===
1. Python
2. Java
3. C++
finish_reason: stop

=== USO DE max_tokens=20 (truncamento por limite de tokens) ===
## A História Detalhada da Inteligência Artificial desde 1950

A história da
finish_reason: length (tokens gerados: 20)

=== USO DE stream=True (recebendo chunks em tempo real) ===
Código que aprende,
Mente fria em telas brilha,
Futuro a chegar.
[Streaming concluído em 20 chunks]


## 1.4 Múltiplas Trocas de Mensagem e a Janela de Contexto

### Por que precisamos reenviar o histórico? (APIs são *Stateless*)

As APIs de LLMs **não guardam estado (são *stateless*)**. Cada requisição HTTP enviada para `chat.completions.create` é completamente independente da anterior.

Se você disser seu nome na primeira chamada e perguntar *"Qual é o meu nome?"* na segunda chamada sem reenviar o histórico, o modelo **não saberá responder**.

Para criar uma conversa de múltiplos turnos, a aplicação precisa manter uma lista de mensagens (`historico`) e, a cada novo turno:
1. Adicionar a nova mensagem `{"role": "user", "content": ...}` ao histórico;
2. Enviar **todo o histórico acumulado** dentro da **Janela de Contexto** do modelo;
3. Receber a resposta e anexá-la como `{"role": "assistant", "content": ...}` ao histórico.

```
Turno 1 enviado à API: [System, User_1]                                 -> Recebe Assistant_1
Turno 2 enviado à API: [System, User_1, Assistant_1, User_2]            -> Recebe Assistant_2
Turno 3 enviado à API: [System, User_1, Assistant_1, User_2, Assistant_2, User_3] -> Recebe Assistant_3
```

In [6]:
# 1º: Demonstrando que chamadas isoladas NÃO possuem memória (Stateless)
r1 = client.chat.completions.create(
    model=MODELO_PADRAO,
    messages=[{"role": "user", "content": "Olá! Meu nome é Carlos e eu trabalho com Engenharia de Dados."}],
)
r2 = client.chat.completions.create(
    model=MODELO_PADRAO,
    messages=[{"role": "user", "content": "Qual é o meu nome e com o que eu trabalho?"}],
)

print("=== CHAMADAS ISOLADAS (SEM HISTÓRICO) ===")
print("Resposta 1:", r1.choices[0].message.content.strip())
print("Resposta 2 (sem contexto anterior):", r2.choices[0].message.content.strip())

=== CHAMADAS ISOLADAS (SEM HISTÓRICO) ===
Resposta 1: Olá Carlos! É um prazer te conhecer. Engenharia de Dados é uma área super dinâmica e crucial no mundo de dados hoje em dia.

Em que tipo de projetos você está focado atualmente? (Ex: ETL/ELT, Data Warehousing, Streaming de dados, MLOps, etc.)

Se tiver alguma dúvida sobre:

* **Tecnologias** (Spark, Kafka, Airflow, dbt, Snowflake, BigQuery, etc.)
* **Melhores práticas** em arquitetura de dados
* **Desafios** em ingestão, qualidade ou governança de dados
* **Tendências** do mercado

...é só perguntar! Como posso te ajudar hoje no seu trabalho ou em algo que você queira discutir sobre a área? 😊
Resposta 2 (sem contexto anterior): Como um modelo de linguagem de inteligência artificial, **eu não tenho acesso a informações pessoais suas**, como seu nome ou a qual trabalho você exerce. Eu não sei quem você é nem o que faz.

**Para que eu possa te ajudar, você precisará me fornecer essas informações.**

**Exemplo:**

* **Se você me disser:

### Gerenciando a Conversa Multi-Turno e Monitorando a Janela de Contexto

Agora vamos implementar um gerenciador de conversa que mantém o histórico completo e registra, a cada turno, quantos tokens de entrada (`prompt_tokens`) estão ocupando a **janela de contexto** do modelo.

Também demonstraremos como aplicar uma **Janela Deslizante (Sliding Window)** para evitar que conversas longas estourem o limite de contexto ou encareçam as chamadas.

In [7]:
class ChatComMemoria:
    """Gerencia o histórico de mensagens e monitora o uso da janela de contexto."""

    def __init__(self, system_prompt: str, max_pares_historico: int | None = None):
        self.system_message = {"role": "system", "content": system_prompt}
        self.historico_dialogo = []  # Armazena mensagens 'user' e 'assistant'
        self.max_pares_historico = max_pares_historico
        self.metricas_turnos = []

    def _montar_janela_contexto(self) -> list[dict]:
        """Monta a lista de mensagens que será enviada à API respeitando a janela deslizante."""
        if self.max_pares_historico is not None:
            # Cada turno tem 2 mensagens (user + assistant), mais a pergunta atual do user
            limite_msgs = self.max_pares_historico * 2 + 1
            dialogo_ativo = self.historico_dialogo[-limite_msgs:]
        else:
            dialogo_ativo = self.historico_dialogo
        return [self.system_message] + dialogo_ativo

    def enviar_mensagem(self, texto_usuario: str) -> str:
        # 1. Adiciona a mensagem do usuário ao histórico
        self.historico_dialogo.append({"role": "user", "content": texto_usuario})

        # 2. Monta o payload que vai para a janela de contexto
        mensagens_enviadas = self._montar_janela_contexto()

        # 3. Chama a API com todo o contexto ativo
        resposta = client.chat.completions.create(
            model=MODELO_PADRAO,
            messages=mensagens_enviadas,
            temperature=0.2,
        )

        texto_assistente = resposta.choices[0].message.content.strip()

        # 4. Salva a resposta do assistente no histórico para os próximos turnos
        self.historico_dialogo.append({"role": "assistant", "content": texto_assistente})

        # 5. Registra métricas de crescimento da janela de contexto
        turno_idx = len(self.metricas_turnos) + 1
        self.metricas_turnos.append({
            "turno": turno_idx,
            "qtd_mensagens_enviadas": len(mensagens_enviadas),
            "prompt_tokens": resposta.usage.prompt_tokens,
            "completion_tokens": resposta.usage.completion_tokens,
            "total_tokens": resposta.usage.total_tokens,
        })
        return texto_assistente


# Simulando uma conversa de 4 turnos mantendo toda a janela de contexto
chat = ChatComMemoria(
    system_prompt="Você é um assistente financeiro objetivo. Responda sempre em no máximo 2 frases."
)

perguntas_conversa = [
    "Olá! Meu nome é Mariana e tenho R$ 2.000 por mês para investir.",
    "Meu perfil é conservador e quero montar uma reserva de emergência.",
    "Quanto eu terei acumulado ao final de 6 meses apenas com os aportes mensais, sem contar juros?",
    "Faça um resumo de 1 linha dizendo meu nome, meu perfil e o valor acumulado em 6 meses.",
]

for pergunta in perguntas_conversa:
    resposta = chat.enviar_mensagem(pergunta)
    metrica = chat.metricas_turnos[-1]
    print(f"--- Turno {metrica['turno']} (Mensagens na janela: {metrica['qtd_mensagens_enviadas']} | Prompt Tokens: {metrica['prompt_tokens']}) ---")
    print(f"Usuário   : {pergunta}")
    print(f"Assistente: {resposta}\n")

print("=== EVOLUÇÃO DO CONSUMO DA JANELA DE CONTEXTO POR TURNO ===")
print(f"{'Turno':<8} | {'Msgs Enviadas':<15} | {'Prompt Tokens (Contexto)':<26} | {'Completion Tokens':<18} | {'Total Tokens'}")
print("-" * 88)
for m in chat.metricas_turnos:
    print(f"{m['turno']:<8} | {m['qtd_mensagens_enviadas']:<15} | {m['prompt_tokens']:<26} | {m['completion_tokens']:<18} | {m['total_tokens']}")

print("\n=== CONTEÚDO COMPLETO DO HISTÓRICO ENVIADO NO ÚLTIMO TURNO ===")
print(json.dumps(chat._montar_janela_contexto(), ensure_ascii=False, indent=2))

--- Turno 1 (Mensagens na janela: 2 | Prompt Tokens: 55) ---
Usuário   : Olá! Meu nome é Mariana e tenho R$ 2.000 por mês para investir.
Assistente: Defina seus objetivos e tolerância a risco para escolher investimentos adequados. Considere diversificar entre renda fixa e variável conforme seu perfil.

--- Turno 2 (Mensagens na janela: 4 | Prompt Tokens: 107) ---
Usuário   : Meu perfil é conservador e quero montar uma reserva de emergência.
Assistente: Priorize investimentos de alta liquidez e baixo risco, como Tesouro Selic ou CDBs de liquidez diária. Mantenha o valor da reserva equivalente a 6 a 12 meses de seus custos fixos.

--- Turno 3 (Mensagens na janela: 6 | Prompt Tokens: 189) ---
Usuário   : Quanto eu terei acumulado ao final de 6 meses apenas com os aportes mensais, sem contar juros?
Assistente: Você acumulará R$ 12.000 ao final de 6 meses, pois 6 meses multiplicados por R$ 2.000 é igual a esse valor. Os juros aumentarão esse montante efetivamente.

--- Turno 4 (Mensagens na

---
# Parte 2: Estratégias de Engenharia de Prompt

A **Engenharia de Prompt** consiste em projetar e estruturar as instruções e o contexto fornecidos ao LLM para maximizar a precisão, confiabilidade e aderência ao formato desejado — sem precisar re-treinar (*fine-tuning*) os pesos do modelo.

A seguir, implementamos e comparamos na prática as principais técnicas utilizadas na indústria.

---

## 2.1 Zero-Shot Prompting

No **Zero-Shot Prompting**, instruímos o modelo a realizar uma tarefa **sem fornecer nenhum exemplo prévio** (*0 exemplos*). O modelo depende exclusivamente do conhecimento adquirido durante seu pré-treinamento e ajuste por instruções (*Instruction Tuning*).

- **Vantagens:** Menor consumo de tokens e rapidez de implementação.
- **Limitações:** Pode não seguir padrões de formatação muito específicos ou regras de negócio idiossincráticas.

In [8]:
prompt_zero_shot = """
Classifique o sentimento das avaliações de clientes abaixo em POSITIVO, NEUTRO ou NEGATIVO.

Avaliações:
1. "A entrega chegou dois dias antes do prazo e a embalagem estava impecável!"
2. "O fone funciona, mas a bateria dura metade do que foi anunciado na caixa."
3. "Recebi o produto hoje pela manhã, ainda vou testar durante a semana."
""".strip()

resp_zero = client.chat.completions.create(
    model=MODELO_PADRAO,
    messages=[{"role": "user", "content": prompt_zero_shot}],
    temperature=0.0,
)
print("=== RESULTADO ZERO-SHOT ===")
print(resp_zero.choices[0].message.content.strip())

=== RESULTADO ZERO-SHOT ===
Aqui está a classificação do sentimento de cada avaliação:

1. **POSITIVO** (A entrega antecipada e a embalagem impecável são pontos muito positivos.)
2. **NEGATIVO** (Embora funcione, a bateria não corresponde à promessa, o que é uma falha de produto/expectativa.)
3. **NEUTRO** (A avaliação é puramente informativa sobre o recebimento e não expressa satisfação ou insatisfação com o uso ainda.)


## 2.2 One-Shot Prompting

No **One-Shot Prompting**, fornecemos **exatamente um exemplo completo (entrada -> saída)** antes do caso real. Isso é especialmente útil quando queremos que o modelo replique uma **convenção de formatação rigorosa** ou estilo específico sem precisarmos escrever longas explicações sobre o formato.

In [9]:
prompt_one_shot = """
Converta descrições informais de bugs em títulos padronizados de commits seguindo o padrão Conventional Commits com tag de severidade.

Exemplo:
Entrada: "Quando o usuário clica em salvar perfil com foto em PNG acima de 2MB, a tela congela."
Saída: fix(perfil)[ALTA]: corrige travamento no upload de imagens PNG maiores que 2MB

Agora processe o caso abaixo seguindo exatamente o mesmo padrão:
Entrada: "O botão de exportar relatório mensal em PDF está com a cor azul desalinhada no modo escuro."
Saída:
""".strip()

resp_one = client.chat.completions.create(
    model=MODELO_PADRAO,
    messages=[{"role": "user", "content": prompt_one_shot}],
    temperature=0.0,
)
print("=== RESULTADO ONE-SHOT ===")
print(resp_one.choices[0].message.content.strip())

=== RESULTADO ONE-SHOT ===
style(relatorio)[BAIXA]: corrige desalinhamento da cor do botão de exportar PDF no modo escuro


## 2.3 Few-Shot Prompting (Aprendizado em Contexto / *In-Context Learning*)

No **Few-Shot Prompting**, fornecemos **múltiplos exemplos ($k$ exemplos, geralmente de 3 a 5)** que demonstram diferentes nuances, casos de borda ou categorias customizadas.

Na API de Chat, podemos passar os exemplos *Few-Shot* de duas formas:
1. Dentro de um único bloco de texto do `user`; ou
2. Como pares explícitos de mensagens `{"role": "user", ...}` e `{"role": "assistant", ...}` no histórico antes da pergunta final (abordagem recomendada em APIs de chat).

In [10]:
# Utilizando pares de mensagens user/assistant para ensinar uma taxonomia interna de tickets bancários
mensagens_few_shot = [
    {
        "role": "system",
        "content": "Você é um classificador automático de ocorrências bancárias. Responda apenas no formato: CATEGORIA | AÇÃO_AUTOMÁTICA",
    },
    # Exemplo 1
    {"role": "user", "content": "Não reconheço uma compra presencial de R$ 4.500 feita agora há pouco em outra cidade."},
    {"role": "assistant", "content": "FRAUDE_CARTAO | BLOQUEAR_CARTAO_IMEDIATAMENTE"},
    # Exemplo 2
    {"role": "user", "content": "Fiz um PIX para minha conta em outra corretora há 2 minutos e o saldo ainda não apareceu."},
    {"role": "assistant", "content": "PROCESSAMENTO_PIX | AGUARDAR_LIQUIDACAO_BACEN"},
    # Exemplo 3
    {"role": "user", "content": "Gostaria de aumentar meu limite diário de transferência noturna de R$ 1.000 para R$ 3.000."},
    {"role": "assistant", "content": "GESTAO_LIMITE | ENCAMINHAR_FLUXO_APP_24H"},
    # Caso Real a ser classificado
    {"role": "user", "content": "Acabei de receber três notificações seguidas de compras internacionais no meu cartão virtual que eu não fiz!"},
]

resp_few = client.chat.completions.create(
    model=MODELO_PADRAO,
    messages=mensagens_few_shot,
    temperature=0.0,
)
print("=== RESULTADO FEW-SHOT (VIA PARES USER/ASSISTANT) ===")
print(resp_few.choices[0].message.content.strip())

=== RESULTADO FEW-SHOT (VIA PARES USER/ASSISTANT) ===
FRAUDE_CARTAO | BLOQUEAR_CARTAO_IMEDIATAMENTE


## 2.4 Role Prompting / System Persona

O **Role Prompting** atribui ao LLM uma identidade profissional, bagagem técnica, público-alvo e restrições operacionais. Ao ativar uma persona específica no `system`, direcionamos o vocabulário, os critérios de análise e o foco da resposta.

In [11]:
proposta_projeto = """
Queremos lançar em 2 semanas um aplicativo médico onde os pacientes enviam fotos de exames laboratoriais
pelo WhatsApp e um LLM envia o diagnóstico completo e receita medicamentos automaticamente sem revisão médica.
""".strip()

personas = {
    "Especialista Jurídico em LGPD e Regulação Médica (CFM)": (
        "Você é um advogado especialista em Direito Digital, LGPD (dados sensíveis de saúde) e normas do Conselho Federal de Medicina. "
        "Avalie a proposta apontando exatamente 3 riscos regulatórios críticos de forma direta."
    ),
    "Arquiteto de Segurança da Informação (AppSec)": (
        "Você é um arquiteto de cibersegurança (CISSP). "
        "Avalie a proposta apontando exatamente 3 riscos técnicos de segurança de dados, prompt injection e alucinação."
    ),
}

for titulo, persona_prompt in personas.items():
    r = client.chat.completions.create(
        model=MODELO_PADRAO,
        messages=[
            {"role": "system", "content": persona_prompt},
            {"role": "user", "content": proposta_projeto},
        ],
        temperature=0.2,
    )
    print(f"=== VISÃO DA PERSONA: {titulo} ===")
    print(r.choices[0].message.content.strip())
    print()

=== VISÃO DA PERSONA: Especialista Jurídico em LGPD e Regulação Médica (CFM) ===
**3 Riscos Regulatórios Críticos:**

1. **Violação da LGPD (Dados Sensíveis de Saúde):** O envio de exames laboratoriais (dados de saúde) via WhatsApp, plataforma não criptografada/controlada pelo hospital/clínica, configura tratamento de dados sensíveis sem a segurança e os mecanismos de consentimento adequados exigidos pela LGPD.
2. **Infração ao CFM (Prática Médica Não Regulamentada):** A emissão de "diagnóstico completo e receita medicamentos automaticamente sem revisão médica" configura exercício ilegal da medicina e viola frontalmente as normas do CFM, que exigem a avaliação clínica humana e a responsabilidade profissional do médico.
3. **Risco de Responsabilidade Civil/Criminal:** A ausência de revisão médica torna o sistema um gerador de erro diagnóstico/terapêutico. Em caso de dano ao paciente, a responsabilidade recairá sobre a operadora do aplicativo, desenvolvedores e, potencialmente, a institu

## 2.5 Saída Estruturada / JSON Mode

Quando integramos LLMs em sistemas de software, bancos de dados ou pipelines de dados, precisamos que a saída seja **interpretável por máquina (JSON)** em vez de texto livre conversacional.

Combinamos:
1. O parâmetro `response_format={"type": "json_object"}` na chamada da API;
2. A especificação explícita do schema JSON desejado no prompt do sistema.

In [12]:
schema_esperado = {
    "cliente": "string",
    "data_evento": "YYYY-MM-DD",
    "itens_extraidos": [
        {"produto": "string", "quantidade": "int", "valor_unitario": "float"}
    ],
    "valor_total_calculado": "float",
    "possui_desconto": "bool",
}

email_desestruturado = """
Oi pessoal do faturamento! Aqui é o Roberto da empresa TechNova Ltda.
Confirmando nosso pedido fechado no dia 15 de março de 2026:
pegamos 3 monitores ultrawide por R$ 1.800,00 cada e 2 teclados mecânicos por R$ 350,00 cada.
Como pagaremos à vista no PIX, vocês prometeram um desconto especial na nota!
""".strip()

resp_json = client.chat.completions.create(
    model=MODELO_PADRAO,
    messages=[
        {
            "role": "system",
            "content": (
                "Extraia as informações do texto e retorne APENAS um objeto JSON válido, sem blocos markdown, "
                f"seguindo estritamente este schema: {json.dumps(schema_esperado, ensure_ascii=False)}"
            ),
        },
        {"role": "user", "content": email_desestruturado},
    ],
    temperature=0.0,
    response_format={"type": "json_object"},
)

conteudo_bruto = resp_json.choices[0].message.content.strip()
dados_python = json.loads(conteudo_bruto)

print("=== JSON RETORNADO E VALIDADO COM json.loads() ===")
print(json.dumps(dados_python, indent=2, ensure_ascii=False))
print(f"\nAcesso direto no Python -> Cliente: {dados_python['cliente']} | Total: R$ {dados_python['valor_total_calculado']}")

=== JSON RETORNADO E VALIDADO COM json.loads() ===
{
  "cliente": "TechNova Ltda.",
  "data_evento": "2026-03-15",
  "itens_extraidos": [
    {
      "produto": "monitores ultrawide",
      "quantidade": 3,
      "valor_unitario": 1800.0
    },
    {
      "produto": "teclados mecânicos",
      "quantidade": 2,
      "valor_unitario": 350.0
    }
  ],
  "valor_total_calculado": 6300.0,
  "possui_desconto": true
}

Acesso direto no Python -> Cliente: TechNova Ltda. | Total: R$ 6300.0


## 2.6 Chain-of-Thought (CoT — Cadeia de Pensamento)

Modelos de linguagem geram texto token por token. Quando pedimos que respondam **imediatamente** a um problema lógico, matemático ou de múltiplas etapas sem escrever os passos intermediários, a probabilidade de erro aumenta porque o modelo não teve "espaço computacional" (tokens intermediários) para decompor o problema.

A técnica **Chain-of-Thought (CoT)** (Wei et al., 2022) instrui o modelo a **explicitar as etapas intermediárias de raciocínio** antes de declarar a resposta final:
- **Zero-Shot CoT:** Adicionar instruções como *"Pense passo a passo antes de responder"*.
- **Few-Shot CoT:** Fornecer exemplos que mostram o raciocínio detalhado até a conclusão.

In [13]:
problema_logico = """
Uma loja de informática tinha 45 notebooks em estoque na segunda-feira.
- Na terça-feira, vendeu 1/3 do estoque inicial e recebeu um lote novo com 18 unidades.
- Na quarta-feira, vendeu metade dos notebooks que estavam disponíveis no início da quarta-feira, mas 3 clientes devolveram 1 notebook cada em perfeito estado no fim do dia.
Quantos notebooks a loja tem disponíveis no final da quarta-feira?
""".strip()

prompt_cot = f"""
Resolva o problema abaixo utilizando Chain-of-Thought (Cadeia de Pensamento):
1. Primeiro, decomponha o estado do estoque dia a dia mostrando cada cálculo intermediário.
2. Ao final, escreva a linha 'RESPOSTA FINAL: <número>'.

Problema:
{problema_logico}
""".strip()

resp_cot = client.chat.completions.create(
    model=MODELO_PADRAO,
    messages=[{"role": "user", "content": prompt_cot}],
    temperature=0.0,
)

print("=== CHAIN-OF-THOUGHT (PASSO A PASSO EXPLÍCITO NO PROMPT) ===")
print(resp_cot.choices[0].message.content.strip())

=== CHAIN-OF-THOUGHT (PASSO A PASSO EXPLÍCITO NO PROMPT) ===
**Chain-of-Thought (Cadeia de Pensamento):**

1. **Estado Inicial (Segunda-feira):** O estoque inicial é de 45 notebooks.
   * Estoque Segunda = 45

2. **Cálculos de Terça-feira:**
   * Vendas na terça: $1/3$ do estoque inicial (45).
     * Vendas = $45 \times (1/3) = 15$ notebooks.
   * Estoque após vendas: $45 - 15 = 30$ notebooks.
   * Recebimento de lote novo: 18 unidades.
   * Estoque no final de terça (Início de quarta): $30 + 18 = 48$ notebooks.

3. **Cálculos de Quarta-feira:**
   * Estoque no início de quarta: 48 notebooks.
   * Vendas na quarta: Metade do estoque disponível no início da quarta.
     * Vendas = $48 / 2 = 24$ notebooks.
   * Estoque após vendas: $48 - 24 = 24$ notebooks.
   * Devoluções no fim do dia: 3 clientes devolveram 1 notebook cada, totalizando 3 notebooks.
   * Estoque no final de quarta: $24 + 3 = 27$ notebooks.

RESPOSTA FINAL: 27


## 2.7 Modelos de Reasoning (Raciocínio Nativo / *Thinking Models*)

Diferentemente do *Chain-of-Thought* tradicional — onde nós pedimos no prompt para um modelo comum escrever os passos na resposta —, os **Modelos de Reasoning** (como a família **NVIDIA Nemotron Reasoning / Ultra**, **DeepSeek-R1**, **OpenAI o1/o3**, **Gemini Thinking**) foram treinados com Aprendizado por Reforço (RL) para realizar uma fase interna de **busca, verificação de hipóteses e autocorreção** antes de produzir a resposta ao usuário.

Na API da NVIDIA NIM, modelos de reasoning retornam:
- `message.reasoning_content`: O fluxo de pensamento interno (deliberação do modelo);
- `message.content`: A resposta final limpa entregue ao usuário;
- `usage.completion_tokens_details.reasoning_tokens`: A contagem de tokens gastos pensando.

Vamos testar isso na prática utilizando o modelo `nvidia/nemotron-3-ultra-550b-a55b`!

In [14]:
desafio_reasoning = """
Três servidores (Alfa, Beta e Gama) processam lotes de dados.
- Juntos, Alfa e Beta processam 70 lotes por hora.
- Juntos, Beta e Gama processam 85 lotes por hora.
- Juntos, Alfa e Gama processam 65 lotes por hora.
Quantos lotes por hora cada servidor processa individualmente? Responda de forma concisa.
""".strip()

resp_reasoning = client.chat.completions.create(
    model=MODELO_REASONING,
    messages=[{"role": "user", "content": desafio_reasoning}],
    temperature=0.2,
    max_tokens=1024,
)

msg_r = resp_reasoning.choices[0].message
pensamento_interno = getattr(msg_r, "reasoning_content", None)

print(f"Modelo utilizado: {MODELO_REASONING}")
print(f"Tokens de prompt: {resp_reasoning.usage.prompt_tokens} | Tokens totais gerados: {resp_reasoning.usage.completion_tokens}")
if resp_reasoning.usage.completion_tokens_details:
    print(f"Tokens gastos em Reasoning interno: {resp_reasoning.usage.completion_tokens_details.reasoning_tokens}")

print("\n=== 1. PENSAMENTO INTERNO DO MODELO (message.reasoning_content) ===")
print(pensamento_interno.strip() if pensamento_interno else "(Sem reasoning_content separado)")

print("\n=== 2. RESPOSTA FINAL ENTREGUE AO USUÁRIO (message.content) ===")
print(msg_r.content.strip())

Modelo utilizado: nvidia/nemotron-3-ultra-550b-a55b
Tokens de prompt: 107 | Tokens totais gerados: 293
Tokens gastos em Reasoning interno: 256

=== 1. PENSAMENTO INTERNO DO MODELO (message.reasoning_content) ===
The user wants to find the individual processing rates of three servers: Alfa (A), Beta (B), and Gama (G).
The system of equations is:
1) A + B = 70
2) B + G = 85
3) A + G = 65

I need to solve for A, B, and G.
Summing all equations: 2A + 2B + 2G = 70 + 85 + 65 = 220
So, A + B + G = 110

Subtracting eq 1 from sum: G = 110 - 70 = 40
Subtracting eq 2 from sum: A = 110 - 85 = 25
Subtracting eq 3 from sum: B = 110 - 65 = 45

Check:
A + B = 25 + 45 = 70 (OK)
B + G = 45 + 40 = 85 (OK)
A + G = 25 + 40 = 65 (OK)

Format the answer concisely.

=== 2. RESPOSTA FINAL ENTREGUE AO USUÁRIO (message.content) ===
**Alfa: 25 lotes/hora**
**Beta: 45 lotes/hora**
**Gama: 40 lotes/hora**


## 2.8 Self-Consistency e Reflexão (Autoavaliação e Refinamento)

Duas estratégias avançadas que elevam significativamente a qualidade e confiabilidade das saídas são:

1. **Self-Consistency (Autoconsistência — Wang et al., 2022):**
   Em vez de gerar um único caminho de raciocínio com `temperature=0`, geramos **múltiplas amostras independentes** com `temperature > 0` (permitindo que o modelo explore diferentes caminhos de raciocínio) e escolhemos a resposta final por **votação majoritária**.
2. **Reflexão / Critique-and-Refine (Gerador $
ightarrow$ Crítico $
ightarrow$ Refinador):**
   O modelo gera uma primeira versão (*rascunho*), em seguida recebe instruções para atuar como um **revisor crítico** procurando falhas, omissões ou vulnerabilidades na própria resposta e, por fim, reescreve a versão definitiva corrigida.

In [15]:
from collections import Counter
import re

# --- Parte A: Self-Consistency (Votação entre múltiplos caminhos de raciocínio) ---
pergunta_sc = """
Um serviço em nuvem custa $0.12 por hora para cada instância ativa.
Se uma empresa rodou 5 instâncias durante 8 horas na manhã e depois escalou para 10 instâncias durante 4 horas à tarde,
qual foi o custo total do dia em dólares?
Pense passo a passo e termine a última linha exatamente com: VALOR_FINAL: <numero>
""".strip()

votos = []
print("=== EXECUTANDO SELF-CONSISTENCY (3 AMOSTRAS DE RACIOCÍNIO) ===")
for i in range(3):
    r = client.chat.completions.create(
        model=MODELO_PADRAO,
        messages=[{"role": "user", "content": pergunta_sc}],
        temperature=0.6,
    )
    texto = r.choices[0].message.content.strip()
    match = re.search(r"VALOR_FINAL:\s*\$?([0-9]+(?:\.[0-9]+)?)", texto)
    valor_extraido = match.group(1) if match else "N/A"
    votos.append(valor_extraido)
    print(f"Amostra {i+1} -> Valor extraído: ${valor_extraido}")

vencedor, contagem = Counter(votos).most_common(1)[0]
print(f"\nConsenso por votação majoritária: ${vencedor} ({contagem}/{len(votos)} votos)")

=== EXECUTANDO SELF-CONSISTENCY (3 AMOSTRAS DE RACIOCÍNIO) ===
Amostra 1 -> Valor extraído: $9.60
Amostra 2 -> Valor extraído: $9.60
Amostra 3 -> Valor extraído: $9.60

Consenso por votação majoritária: $9.60 (3/3 votos)


In [16]:
# --- Parte B: Reflexão (Gerador -> Crítico -> Refinador) ---
tarefa_codigo = (
    "Escreva uma função Python simples `buscar_usuario(conn, username)` que consulta a tabela `usuarios` "
    "em um banco SQLite pelo `username`."
)

# Passo 1: Geração Inicial (propositalmente pedindo um rascunho rápido)
rascunho = client.chat.completions.create(
    model=MODELO_PADRAO,
    messages=[
        {"role": "user", "content": f"{tarefa_codigo} Mostre apenas o código básico usando f-string na query para fins de exemplo inicial."}
    ],
    temperature=0.0,
).choices[0].message.content.strip()

# Passo 2: Crítica / Reflexão de Segurança e Boas Práticas
critica = client.chat.completions.create(
    model=MODELO_PADRAO,
    messages=[
        {
            "role": "system",
            "content": "Você é um revisor de código focado em segurança (OWASP) e boas práticas Python. Liste de forma curta os problemas do código.",
        },
        {"role": "user", "content": f"Avalie criticamente este código:\n{rascunho}"},
    ],
    temperature=0.0,
).choices[0].message.content.strip()

# Passo 3: Refinamento baseado na Crítica
versao_refinada = client.chat.completions.create(
    model=MODELO_PADRAO,
    messages=[
        {
            "role": "user",
            "content": (
                f"Código original:\n{rascunho}\n\n"
                f"Crítica de segurança:\n{critica}\n\n"
                "Reescreva a função corrigindo todas as falhas apontadas (use queries parametrizadas, type hints e tratamento básico). "
                "Mostre apenas o código corrigido e 1 frase explicativa."
            ),
        }
    ],
    temperature=0.0,
).choices[0].message.content.strip()

print("=== 1. RASCUNHO INICIAL ===")
print(rascunho)
print("\n=== 2. AUTOAVALIAÇÃO / CRÍTICA DE SEGURANÇA ===")
print(critica)
print("\n=== 3. VERSÃO REFINADA APÓS REFLEXÃO ===")
print(versao_refinada)

=== 1. RASCUNHO INICIAL ===
Aqui está a função Python simples `buscar_usuario(conn, username)` usando f-string para construir a query SQL, conforme solicitado para fins de exemplo inicial.

**⚠️ Aviso de Segurança Importante:** **NUNCA** use f-strings (ou concatenação de strings) para inserir dados de usuário diretamente em queries SQL em um ambiente de produção. Isso é extremamente vulnerável a ataques de **SQL Injection**. O método correto é sempre usar *placeholders* (`?`) e passar os parâmetros separadamente para o método `cursor.execute()`.

```python
import sqlite3

def buscar_usuario(conn: sqlite3.Connection, username: str) -> tuple | None:
    """
    Consulta a tabela 'usuarios' pelo username usando f-string (APENAS PARA EXEMPLO INICIAL).

    Args:
        conn: Conexão ativa com o banco de dados SQLite.
        username: O nome de usuário a ser buscado.

    Returns:
        Uma tupla contendo os dados do usuário ou None se não encontrado.
    """
    cursor = conn.cursor()


## 2.9 Prompt Chaining e Padrão ReAct (Reason + Act)

### Prompt Chaining (Encadeamento de Prompts)
Em vez de criar um único "megaprompt" complexo que tenta fazer tudo de uma só vez, dividimos o fluxo em **etapas sequenciais especializadas**, onde a saída da Etapa $N$ serve de entrada para a Etapa $N+1$.

### Padrão ReAct (Reason + Act — Yao et al., 2022)
Base dos **Agentes de IA modernos**: o modelo alterna entre:
1. **Thought (Pensamento):** Raciocina sobre o que precisa descobrir;
2. **Action (Ação):** Solicita a execução de uma ferramenta externa (ex.: consultar banco de dados, API de clima ou calculadora);
3. **Observation (Observação):** Recebe o resultado real da ferramenta executada pelo Python e formula a resposta final com precisão absoluta.

In [17]:
# --- Exemplo Prático do Padrão ReAct (LLM + Ferramenta Python) ---

# 1. Definimos nossas ferramentas reais no Python
TABELA_FRETE_REGIAO = {
    "SUDESTE": 25.00,
    "SUL": 35.00,
    "NORDESTE": 55.00,
    "NORTE": 70.00,
    "CENTRO-OESTE": 40.00,
}


def calcular_pedido_com_frete(valor_produtos: float, regiao: str, cupom_desconto_pct: float = 0.0) -> dict:
    """Ferramenta Python que calcula o valor exato com desconto e frete regional."""
    regiao_norm = regiao.upper().strip()
    frete = TABELA_FRETE_REGIAO.get(regiao_norm, 50.0)
    valor_com_desconto = round(valor_produtos * (1 - cupom_desconto_pct / 100.0), 2)
    total_final = round(valor_com_desconto + frete, 2)
    return {
        "valor_original": valor_produtos,
        "desconto_aplicado_pct": cupom_desconto_pct,
        "valor_com_desconto": valor_com_desconto,
        "regiao": regiao_norm,
        "frete": frete,
        "total_final": total_final,
    }


# 2. Prompt do Agente ReAct
system_react = """
Você opera no ciclo ReAct (Thought -> Action).
Você tem acesso à ferramenta:
- calcular_pedido_com_frete(valor_produtos: float, regiao: str, cupom_desconto_pct: float)

Quando precisar calcular o valor de um pedido, responda APENAS em JSON com os campos:
{
  "thought": "seu raciocínio sobre quais parâmetros extrair da pergunta do usuário",
  "action": "calcular_pedido_com_frete",
  "action_input": {"valor_produtos": <float>, "regiao": "<REGIAO>", "cupom_desconto_pct": <float>}
}
""".strip()

pergunta_cliente = (
    "Olá! Quero comprar um kit de sensores que custa R$ 480,00 para entrega em Recife (região Nordeste) "
    "e tenho um cupom de 15% de desconto nos produtos. Qual fica o valor total com o frete?"
)

# Passo 1 do ReAct: LLM analisa a pergunta (Thought) e decide qual ação invocar (Action)
decisao_llm = client.chat.completions.create(
    model=MODELO_PADRAO,
    messages=[
        {"role": "system", "content": system_react},
        {"role": "user", "content": pergunta_cliente},
    ],
    temperature=0.0,
    response_format={"type": "json_object"},
)

plano_acao = json.loads(decisao_llm.choices[0].message.content)
print("=== ETAPA 1 DO REACT (THOUGHT + ACTION) ===")
print(json.dumps(plano_acao, indent=2, ensure_ascii=False))

# Passo 2 do ReAct: O ambiente Python executa a ferramenta e gera a Observation
observacao = calcular_pedido_com_frete(**plano_acao["action_input"])
print("\n=== ETAPA 2 DO REACT (OBSERVATION DA FERRAMENTA PYTHON) ===")
print(json.dumps(observacao, indent=2, ensure_ascii=False))

# Passo 3 do ReAct: O LLM recebe a Observation e redige a resposta final ao cliente
resposta_final_react = client.chat.completions.create(
    model=MODELO_PADRAO,
    messages=[
        {"role": "system", "content": "Responda ao cliente de forma cordial e direta usando os dados exatos da observação."},
        {"role": "user", "content": pergunta_cliente},
        {"role": "assistant", "content": json.dumps(plano_acao, ensure_ascii=False)},
        {"role": "user", "content": f"OBSERVATION: {json.dumps(observacao, ensure_ascii=False)}"},
    ],
    temperature=0.2,
)

print("\n=== ETAPA 3 DO REACT (FINAL ANSWER AO USUÁRIO) ===")
print(resposta_final_react.choices[0].message.content.strip())

=== ETAPA 1 DO REACT (THOUGHT + ACTION) ===
{
  "thought": "O usuário deseja calcular o valor total de um pedido. Os parâmetros fornecidos são: valor dos produtos (R$ 480,00), região de entrega (Recife, que pertence à região Nordeste) e cupom de desconto (15%). Devo usar a ferramenta calcular_pedido_com_frete com esses valores.",
  "action": "calcular_pedido_com_frete",
  "action_input": {
    "valor_produtos": 480.0,
    "regiao": "Nordeste",
    "cupom_desconto_pct": 15.0
  }
}

=== ETAPA 2 DO REACT (OBSERVATION DA FERRAMENTA PYTHON) ===
{
  "valor_original": 480.0,
  "desconto_aplicado_pct": 15.0,
  "valor_com_desconto": 408.0,
  "regiao": "NORDESTE",
  "frete": 55.0,
  "total_final": 463.0
}

=== ETAPA 3 DO REACT (FINAL ANSWER AO USUÁRIO) ===
Olá! O valor total do kit de sensores para entrega em Recife (Nordeste), com o desconto de 15%, é de **R$ 463,00**.

(Valor com desconto: R$ 408,00 + Frete: R$ 55,00 = R$ 463,00)


---
## Resumo Comparativo das Estratégias de Engenharia de Prompt

| Estratégia | Quando Utilizar | Custo de Tokens | Complexidade |
| :--- | :--- | :---: | :---: |
| **Zero-Shot** | Tarefas simples, diretas e bem conhecidas (tradução, resumo, classificação padrão). | Baixo | Baixa |
| **One-Shot** | Garantir que a resposta siga um formato ou template específico. | Baixo/Médio | Baixa |
| **Few-Shot** | Tarefas com regras de negócio próprias, taxonomias internas ou casos de borda. | Médio | Média |
| **Role / System Persona** | Ajustar tom de voz, profundidade técnica, rigor de análise e restrições comportamentais. | Baixo | Baixa |
| **JSON Mode / Saída Estruturada** | Integração com APIs, bancos de dados e automações que exigem parse determinístico. | Baixo/Médio | Média |
| **Chain-of-Thought (CoT)** | Problemas lógicos, matemáticos ou decisões que dependem de etapas intermediárias. | Médio/Alto | Média |
| **Modelos de Reasoning** | Resolução de problemas complexos, inferência profunda, planejamento e verificação automática. | Alto | Baixa no prompt |
| **Self-Consistency / Reflexão** | Tarefas críticas onde é necessário reduzir alucinações ou auditar segurança/qualidade. | Alto (múltiplas chamadas) | Alta |
| **Prompt Chaining / ReAct** | Fluxos de trabalho em múltiplas etapas ou agentes que consultam ferramentas externas. | Médio/Alto | Alta |